In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CARREGAMENTO DOS DADOS E MAPEAMENTO
# ==============================================================================
file_path = 'TELESAPFormulrioMdic-EXPORTAOBOLSISTAAtua_DATA_LABELS_2025-07-23_1329.csv'
df = pd.read_csv(file_path, low_memory=False)

col_genero = next((c for c in df.columns if 'identidade de gênero' in c.lower()), 'Identidade de gênero')
col_sexo = next((c for c in df.columns if 'sexo' in c.lower()), 'Sexo')

cols_inf = [c for c in df.columns if 'doenças infecciosas' in c.lower() and 'choice=' in c.lower() and 'não' not in c.lower()]
cols_cro = [c for c in df.columns if 'doenças crônicas' in c.lower() and 'choice=' in c.lower() and 'não' not in c.lower()]
cols_esp = [c for c in df.columns if ('encaminhamento' in c.lower() or 'especialidade' in c.lower()) and 'choice=' in c.lower()]
cols_ciap = [c for c in df.columns if 'ciap' in c.lower()]

# ==============================================================================
# 2. TRATAMENTO LONGITUDINAL E IMPUTAÇÃO DE GÊNERO
# ==============================================================================
df = df.sort_values(['Record ID', 'Repeat Instance'], na_position='first')
cols_to_fill = [col_genero, col_sexo] + cols_inf + cols_cro + cols_esp
cols_to_fill = [c for c in cols_to_fill if c and c in df.columns]

df[cols_to_fill] = df.groupby('Record ID')[cols_to_fill].ffill()
df_ev = df[df['Repeat Instrument'].notna()].copy()

def mapear_genero_inteligente(row):
    gen = str(row.get(col_genero, '')).lower().strip()
    sexo = str(row.get(col_sexo, '')).lower().strip()
    
    if gen == 'nan' or gen == '' or 'prefere não' in gen:
        if 'masculino' in sexo or 'homem' in sexo or sexo == 'm': return 'Homem Cis'
        if 'feminino' in sexo or 'mulher' in sexo or sexo == 'f': return 'Mulher Cis'
        return None
    
    if 'homem transgênero' in gen: return 'Homem Trans'
    if 'mulher transgênero' in gen or 'travesti' in gen: return 'Mulher Trans'
    if 'homem cisgênero' in gen: return 'Homem Cis'
    if 'mulher cisgênero' in gen: return 'Mulher Cis'
    return None

df_ev['Grupo_Genero'] = df_ev.apply(mapear_genero_inteligente, axis=1)
df_plot = df_ev.dropna(subset=['Grupo_Genero']).copy()
df_plot['Carga Total de Doenças'] = (df_plot[cols_inf] == 'Checked').sum(axis=1) + (df_plot[cols_cro] == 'Checked').sum(axis=1)

# ==============================================================================
# 3. PREPARAÇÃO (MELTING) PARA AS ESPECIALIDADES
# ==============================================================================
df_melted = df_plot.melt(
    id_vars=['Record ID', 'Grupo_Genero', 'Carga Total de Doenças'],
    value_vars=cols_esp,
    var_name='Especialidade',
    value_name='Marcado'
)

df_melted = df_melted[df_melted['Marcado'] == 'Checked'].copy()
df_melted['Especialidade'] = df_melted['Especialidade'].apply(lambda x: x.split('=')[-1].replace(')', '').strip())

especialidades_alvo = set()
for grupo in df_melted['Grupo_Genero'].unique():
    top_grupo = df_melted[df_melted['Grupo_Genero'] == grupo]['Especialidade'].value_counts().head(3).index
    especialidades_alvo.update(top_grupo)

for esp in df_melted['Especialidade'].unique():
    if 'psiquiatria' in esp.lower() or 'psicologia' in esp.lower() or 'saúde mental' in esp.lower():
        especialidades_alvo.add(esp)

df_final = df_melted[df_melted['Especialidade'].isin(especialidades_alvo)]
ordem_especialidades = sorted(list(especialidades_alvo))

# ==============================================================================
# 4. EXPORTAÇÃO DAS BASES (CSV) PARA O ARTIGO
# ==============================================================================
# Base 1: Complexidade
tabela_complexidade = df_final.groupby(['Especialidade', 'Grupo_Genero'])['Carga Total de Doenças'].agg(['mean', 'std', 'count']).reset_index()
tabela_complexidade.columns = ['Especialidade', 'Identidade de Gênero', 'Média de Doenças', 'Desvio Padrão', 'N (Atendimentos)']
tabela_complexidade.to_csv('base_complexidade_genero.csv', index=False, encoding='utf-8-sig')

# Base 2: CIAP
dados_ciap = []
if cols_ciap:
    col_ciap_alvo = cols_ciap[0]
    for grupo in ['Mulher Cis', 'Homem Cis', 'Mulher Trans', 'Homem Trans']:
        sub = df_plot[df_plot['Grupo_Genero'] == grupo]
        top_ciaps = sub[col_ciap_alvo].dropna().value_counts().head(5) 
        
        for ciap, contagem in top_ciaps.items():
            dados_ciap.append({
                'Identidade de Gênero': grupo,
                'N Total (Grupo)': len(sub),
                'Motivo CIAP-2': str(ciap).strip(),
                'Frequência': contagem,
                'Proporção (%)': round((contagem / len(sub) * 100), 1) if len(sub) > 0 else 0
            })
            
    pd.DataFrame(dados_ciap).to_csv('base_ciap_por_genero.csv', index=False, encoding='utf-8-sig')

# ==============================================================================
# 5. RELATÓRIO NO TERMINAL (CIAP-2 POR GÊNERO)
# ==============================================================================
print(f"\n{'='*90}")
print("PRINCIPAIS MOTIVOS DE CONSULTA (CIAP-2) POR IDENTIDADE DE GÊNERO")
print(f"{'='*90}")

if not cols_ciap:
    print("Nenhuma coluna contendo o termo 'CIAP' foi encontrada.")
else:
    col_ciap_alvo = cols_ciap[0]
    grupos = ['Mulher Cis', 'Homem Cis', 'Mulher Trans', 'Homem Trans']

    for grupo in grupos:
        sub = df_plot[df_plot['Grupo_Genero'] == grupo]
        print(f"\n[{grupo.upper()}] - Atendimentos totais da amostra (após imputação): {len(sub)}")

        if len(sub) > 0:
            top_ciaps = sub[col_ciap_alvo].dropna().value_counts().head(3)
            if len(top_ciaps) > 0:
                for ciap, contagem in top_ciaps.items():
                    print(f"  -> {str(ciap)[:60]:<60} ({contagem} casos)")
            else:
                print("  -> Sem CIAP registrado para os pacientes deste grupo.")
print(f"\n{'='*90}")

# ==============================================================================
# 6. VISUALIZAÇÃO GRÁFICA (BARPLOT)
# ==============================================================================
plt.figure(figsize=(14, 10))
sns.set_theme(style="whitegrid")

paleta = {'Mulher Cis': '#95a5a6', 'Homem Cis': '#34495e', 'Mulher Trans': '#9b59b6', 'Homem Trans': '#e67e22'}

ax = sns.barplot(
    data=df_final,
    y="Especialidade",
    x="Carga Total de Doenças",
    hue="Grupo_Genero",
    palette=paleta,
    capsize=.1,
    err_kws={'linewidth': 1.5},
    order=ordem_especialidades,
    hue_order=['Mulher Cis', 'Homem Cis', 'Mulher Trans', 'Homem Trans']
)

plt.title("Complexidade Clínica (Média de Doenças) por Especialidade e Gênero", fontsize=16, fontweight='bold', pad=20)
plt.xlabel("Média de Doenças Detectadas no Paciente (Infecciosas + Crônicas)", fontsize=13, fontweight='bold')
plt.ylabel("Especialidade Médica Requisitada", fontsize=13, fontweight='bold')
sns.despine(left=True, bottom=True)
plt.legend(title="Identidade de Gênero", bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False, fontsize=11)
plt.tight_layout()
plt.show()